##Prepare the dataset

In [ ]:
# download dependencies
!git clone https://github.com/attardi/wikiextractor.git
!pip install -q transformers sentencepiece opencc-python-reimplemented

Cloning into 'wikiextractor'...
remote: Enumerating objects: 771, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 771 (delta 17), reused 14 (delta 14), pack-reused 748 (from 2)
Receiving objects: 100% (771/771), 1.31 MiB | 15.44 MiB/s, done.
Resolving deltas: 100% (450/450), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 30.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import random
import shutil

from tqdm import tqdm
from opencc import OpenCC

from transformers import AutoTokenizer

In [ ]:
# use python 3.10 for WikiExtractor
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-venv

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Hit:13 https://ppa.launchpadcontent.net/gra

In [ ]:
from google.colab import drive

# get access to drive
drive.mount('/content/drive')

# wiki filepath
WIKI_PATH = '/content/drive/MyDrive/Colab_Notebooks/Advanced_CL/final/codes/zhwiki-20260401-pages-articles-multistream.xml'
OUTPUT_PATH = '/content/zhwiki_subset.xml'

Mounted at /content/drive


In [ ]:
# extract 30000 files
max_pages = 30000
count = 0

with open(WIKI_PATH, "r", encoding="utf-8", errors="ignore") as fin:
    with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:

        for line in fin:
            fout.write(line)

            # count the number of files
            if "</page>" in line:
                count += 1

                if count % 2000 == 0:
                    print(count)

                if count >= max_pages:
                    fout.write("</mediawiki>\n")
                    break

print("done:", count)

2000
4000
6000
8000
10000
12000
14000
16000
18000
20000
22000
24000
26000
28000
30000
done: 30000


In [ ]:
# run WikiExtractor & save output .json file
# output: /content/extract

%cd /content/wikiextractor

!python3.10 -m wikiextractor.WikiExtractor \
--json \
--processes 2 \
-o /content/extracted \
'/content/zhwiki_subset.xml'

/content/wikiextractor
INFO: Preprocessing '/content/zhwiki_subset.xml' to collect template definitions: this may take some time.
INFO: Loaded 379 templates in 4.1s
INFO: Starting page extraction from /content/zhwiki_subset.xml.
INFO: Using 2 extract processes.
INFO: Finished 2-process extraction of 25439 articles in 83.1s (306.3 art/s)


In [ ]:
# initialise OpenCC to convert traditional Chinese to simplified Chinese
converter = OpenCC('t2s')

# initialise Qwen tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen1.5-0.5B",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# paragraph length set to 80 tokens
MIN_LEN = 128
MAX_LEN = 128

# train, val, test set size
TRAIN_SIZE = 1000  # 1000 paragraphs
VAL_SIZE = 200   # 200 paragraphs
TEST_SIZE = 200   # 200 paragraphs

In [ ]:
# define a function to extract paragraphs
def extract_chunk(text):
  # convert traditional Chinese to simplified Chinese
  text = converter.convert(text)

  # split the text into paragraphs
  paragraphs = text.split('\n')

  # select paragraphs with more than 128 characters
  paragraphs = [p.strip() for p in paragraphs if len(p.strip()) > 128]

  # determine if the paragraphs list is empty
  if len(paragraphs) == 0:
    return None

  l = len(paragraphs) - 1
  num = random.randint(0, l)

  # select a random paragraph
  para = paragraphs[num]

  # tokenizing & convert token to index
  tokens = tokenizer(para, add_special_tokens = False)['input_ids']

  # determine if the paragraph has more than 128 tokens
  if len(tokens) < MIN_LEN:
    return None

  # extract first 128 tokens of the paragraph
  tokens = tokens[:MAX_LEN]

  # convert the index to tokens
  chunk = tokenizer.decode(tokens)

  return chunk

In [ ]:
# load the output json file of WikiExtractor
# preallocate a list to restore data
dataset = []
# output path
base_dir = "/content/extracted"
# initialise error counter
error = 0

# loop through all the files
for subdir in tqdm(os.listdir(base_dir)):
    # get filepaths
    subdir_path = os.path.join(base_dir, subdir)

    # determine whether it is a valid path
    if not os.path.isdir(subdir_path):
        continue

    # loop through all the files
    for filename in os.listdir(subdir_path):
        # get filepath
        file_path = os.path.join(subdir_path, filename)

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                # loop through all the lines
                for line in f:
                    # load data
                    data = json.loads(line)
                    # convert title from traditional Chinese to simplified Chinese
                    title = converter.convert(
                        data["title"]
                    )
                    # extract text
                    text = data["text"]
                    # extract suitable paragraph & slice to 80 tokens
                    chunk = extract_chunk(text)

                    # determine if the chunk is empty
                    if chunk is None:
                        continue

                    # add data to the dataset
                    dataset.append({
                        "title": title,
                        "text": chunk
                    })

        except Exception as e:

            print(e)

            error += 1


# inspect data
print("\nTotal number of samples:", len(dataset))
print("Total number of errors:", error)

100%|██████████| 3/3 [01:54<00:00, 38.15s/it]


Total number of samples: 6083
Total number of errors: 0


In [ ]:
# split the data into training, validation & test sets

# randomly shuffle the data
random.shuffle(dataset)

# slice the dataset
train_data = dataset[:TRAIN_SIZE] # training set
val_data = dataset[TRAIN_SIZE:(TRAIN_SIZE+VAL_SIZE)]  # validation set
test_data = dataset[(TRAIN_SIZE+VAL_SIZE):(TRAIN_SIZE+VAL_SIZE+TEST_SIZE)]  # test set

# inspect
print("Training set size:", len(train_data))
print("Validation set size:", len(val_data))
print("Test set size:", len(test_data))

Training set size: 1000
Validation set size: 200
Test set size: 200


In [ ]:
# define a function to save jsonl file
def save_jsonl(data, filename):
  with open(filename, 'w', encoding='utf-8') as f:
    for sample in data:
      f.write(
          json.dumps(sample, ensure_ascii=False) # ensure_ascii=False because they're Chinese characters
          + "\n"
      )


# save jsonl files
save_jsonl(train_data, 'train.jsonl')
save_jsonl(val_data, 'val.jsonl')
save_jsonl(test_data, 'test.jsonl')

# inspect the jsonl files
print("Training:")
with open('train.jsonl', 'r', encoding='utf-8') as f:
  for i in range(3):
    print(f.readline())
print('--------------------------------------')

print('\nValidation:')
with open('val.jsonl', 'r', encoding='utf-8') as f:
  for i in range(3):
    print(f.readline())
print('--------------------------------------')

print('\nTest:')
with open('test.jsonl', 'r', encoding='utf-8') as f:
  for i in range(3):
    print(f.readline())

Training:
{"title": "功夫 (2004年电影)", "text": "《功夫》于2004年多伦多国际电影节首映，接下来于2004年12月，在中国内地、香港上映。该片第一次于美国放映是在2005年1月的日舞影展，并于2005年4月22日在全美上映。最晚的是欧洲，在2005年6月上映。功夫在香港的分级为第ⅡB级（青少年及儿童不宜）、R级于美国，是由于内容有过多的暴力武打，在其他国家也限定13岁到18岁需家人陪看。"}

{"title": "哈勃空间望远镜", "text": "哈伯望远镜上最初的两台主电脑是中央处理器时脉频率1.25 MHz、由洛克威尔公司自动化部门开发的DF-224电脑系统，这款电脑带有三个冗余CPU跟两个冗余的、使用二极体—电晶体逻辑的、由西屋电气及高达德太空飞行中心开发的（美国太空总署标准太空载具电脑第一型）系统。在1993年的维护任务1（SM1）中，这款电脑被加上了一个共同处理器，这共同处理气包含了两序列的冗余，每个冗余各有一个"}

{"title": "梁朝伟", "text": "2002年，梁朝伟宣传电影《英雄》期间，接受香港英文杂志《B International》采访，提到片中刺客没有刺杀秦始皇，是因为他渴望和平，不想天下大乱。梁朝伟借古喻今，由此谈到对「六四」事件的看法：「在六四事件期间，我没有参加任何示威游行，因为中国政府的行径是正确的，维护稳定对每个人都有好处。」该言论引发香港哗然。梁朝伟随后称被杂志断章取义，《B International》否认断章取义。梁朝伟在"}

--------------------------------------

Validation:
{"title": "荣德生", "text": "抗战胜利后，荣德生两次遭绑架，被勒索款项达百万美元，发生在高恩路（今高安路）荣德生住宅门前的一次被绑架案，是在1946年（民国35年）4月25日。那天，荣德生准备去总公司，离家门不远即被数名穿制服匪徒架上汽车而去，他们使用的是国军第三方面军司令部的“逮捕证”和淞沪警备司令部的汽车，当时，舆论哗然，认为是军事机关与匪徒串通作案，蒋介石严"}

{"title": "驻马店市", "text": "古为豫州之地。西周武王灭商后，封其五弟叔度于蔡。武王死后发生三监之乱，周公旦打败蔡叔度。周成

In [ ]:
# save jsonl files to drive
SAVE_DIR = '/content/drive/MyDrive/Colab_Notebooks/Advanced_CL/final/codes'
save_jsonl(train_data, f"{SAVE_DIR}/train.jsonl")
save_jsonl(val_data, f"{SAVE_DIR}/val.jsonl")
save_jsonl(test_data, f"{SAVE_DIR}/test.jsonl")